# Instances Mailout Notebook

This notebook fetches OpenStack instances and sends notifications to project managers and members.

In [ ]:
from mailout_helper import MailoutHelper

## Setup MailoutHelper

Create a Mailout Helper instance

In [ ]:
helper = MailoutHelper()

## Set start/end times for a scheduled outage

This is *optional* if your mailout is for an outage

In [ ]:
helper.set_times(
    start_time = '2026-03-01 09:00:00',
    end_time = '2026-03-01 17:00:00',
    timezone = 'Australia/Melbourne',
)

## Setup OpenStack

In [ ]:
helper.setup_openstack('https://identity.rc.nectar.org.au/')

## Fetch OpenStack Instances

This is an example of getting instances for a specific project, but you could use some other API query to get all instances on a Hypervisor for example.

See https://docs.openstack.org/api-ref/compute/?expanded=list-servers-detail#list-server-request for a list of applicable filters that can be used for listing servers.

In [ ]:
# Fetch instances from OpenStack for a project
project_id = 'f42f9588576c43969760d81384b83b1f'  # NeCTAR-Admins
instances = list(helper.conn.compute.servers(all_tenants=True, project_id=project_id))
print(f"Found {len(instances)} instances")

## Populate Project and User Data

In [ ]:
# Group instances by project and fetch user information
data = helper.populate_data_from_instances(instances)
print(f"Found {len(data)} projects with instances")

## Generate Notifications

In [ ]:
# Subject for notfications
subject = 'Test notification for project {{ project.name }}'

# Body template filename for the notifications
body = 'host-planned-outage-notification.j2'


notifications = helper.generate_notifications_from_instances(
    data,
    subject,
    body,
)

print(f"Generated {len(notifications)} notifications")

## Preview Notifications

In [ ]:
# Preview a notification
helper.preview_notification(notifications[0])

## Send Notifications

In [ ]:
# Are you ready to send these?
for notification in notifications[3:4]:
    print(f"Sending notification to {notification['to']} and {len(notification.get('cc', None))} others...")
    #helper.send_notification(notification)

print(f"Finished")